In [315]:
import pandas as pd
CSV_PATH = "../raw/2026-06-17_match_data.csv"
df = pd.read_csv(CSV_PATH)
row = df[df["event_id"] == 15186687]
row.head()
# 15186709  15186722  15186504  15186687

,event_id,incidents,best_players_summary,lineups,graph,highlights,average_positions,statistics,shotmap,competition,kickoff,home_team,home_team_id,away_team,away_team_id,home_score,away_score,slug,custom_id,sofascore_link
3,15186687,"{""incidents"": [{""text"": ""FT"", ""homeScore"": 1, ...","{""bestHomeTeamPlayers"": [{""value"": ""8.2"", ""lab...","{""confirmed"": true, ""home"": {""players"": [{""pla...","{""graphPoints"": [{""minute"": 1, ""value"": -2}, {...","{""highlights"": [{""title"": ""Ghana - Panama "", ""...","{""home"": [{""player"": {""name"": ""Lawrence Ati Zi...","{""statistics"": [{""period"": ""ALL"", ""groups"": [{...","{""shotmap"": [{""player"": {""name"": ""Ismael D\u00...",FIFA World Cup 2026,2026-06-18 00:00,Ghana,4764,Panama,5164,1,0,panama-ghana,oVbsodc,https://www.sofascore.com/fr/football/match/pa...


In [ ]:
#match (event_id, competition, kickoff, home_team, home_team_id, away_team, away_team_id, home_score, away_score, slug, custom_id, sofascore_link, highilight:(highlights.loc[highlights['subtitle'] == 'Full Highlights'], 'url'))
#team (team_id, teamName)
#match_team (event_id, team_id, teamName, isHome, score, stats: from home_stats or away_stats, team_formation)
#players just like in players
#match_players (event_id, team_id, idPlayer, and stats from final_home_players or final_away_players based on the match_team.isHome + averageX and averageY from average Positions)
#goals_match_team, cards_match_team, subtitutions_match_team (from goals, cards, substitutions, join event_id, team_id where team_id.isHome = thing.isHome)
#highlights (just like highlights add just even_id)
#shotmaps just like shotmaps (eventId, teamId, playerId instead of Player)

In [87]:
import json
incidents_json = json.loads(row["incidents"])
incidents = pd.DataFrame(incidents_json["incidents"])
incidents['incidentType']

0           period
1             card
2       injuryTime
3             card
4     substitution
5     substitution
6     substitution
7     substitution
8     substitution
9     substitution
10    substitution
11    substitution
12    substitution
13          period
14            goal
15      injuryTime
16            card
17            card
18            goal
Name: incidentType, dtype: object

In [88]:
substitustions = incidents.loc[incidents['incidentType']=='substitution' ,['isHome','playerIn', 'playerOut', 'injury', 'time', 'addedTime']]
cards = incidents.loc[incidents['incidentType']=='substitution', ['isHome', 'incidentClass', 'time', 'addedTime']]
goals = incidents.loc[incidents['incidentType']=='goal', ['isHome', 'homeScore', 'awayScore', 'time', 'addedTime', 'player', 'assist1', 'footballPassingNetworkAction']]

In [143]:
final_cols = [
    'player', 'teamId', 'jerseyNumber', 'position', 'substitute', 'captain',
    'totalPass', 'accuratePass', 'totalLongBalls', 'accurateLongBalls',
    'goalAssist', 'accurateOwnHalfPasses', 'totalOwnHalfPasses',
    'totalOppositionHalfPasses', 'totalClearance', 'ballRecovery',
    'goodHighClaim', 'totalKeeperSweeper', 'accurateKeeperSweeper',
    'minutesPlayed', 'touches', 'possessionLostCtrl', 'expectedAssists',
    'totalBallCarriesDistance', 'ballCarriesCount', 'totalProgression',
    'progressiveBallCarriesCount', 'totalShots', 'goalsPrevented',
    'passValueNormalized', 'dribbleValueNormalized',
    'defensiveValueNormalized', 'goalkeeperValueNormalized',
    'accurateOppositionHalfPasses', 'totalCross', 'aerialLost', 'aerialWon',
    'duelLost', 'duelWon', 'totalContest', 'wonContest', 'interceptionWon',
    'lastManTackle', 'totalTackle', 'wonTackle', 'unsuccessfulTouch',
    'fouls', 'topSpeed', 'kilometersCovered', 'numberOfSprints',
    'metersCoveredRunningKm', 'metersCoveredHighSpeedRunningKm',
    'metersCoveredSprintingKm', 'outfielderBlock', 'keyPass', 'dispossessed',
    'shotOffTarget', 'expectedGoals', 'bestBallCarryProgression',
    'totalProgressiveBallCarriesDistance', 'shotValueNormalized',
    'accurateCross', 'challengeLost', 'bigChanceCreated', 'wasFouled',
    'blockedScoringAttempt', 'onTargetScoringAttempt',
    'expectedGoalsOnTarget', 'goals', 'totalOffside', 'saves',
    'keeperSaveValue', 'errorLeadToAGoal', 'savedShotsFromInsideTheBox',
    'bigChanceMissed', 'hitWoodwork', 'errorLeadToAShot', 'penaltyFaced',
    'punches', 'penaltyConceded', 'penaltyWon'
]

In [ ]:
lineups = json.loads(row["lineups"].iloc[0])
lineups = pd.DataFrame(lineups)
team_formations = [lineups.loc['formation', 'home'], lineups.loc['formation', 'away']]
home_players = pd.DataFrame(lineups.loc['players', 'home'])
away_players = pd.DataFrame(lineups.loc['players', 'away'])
home_stats_df = pd.json_normalize(home_players['statistics'])
home_stats_df.index = home_players.index
home_players = home_players.drop(columns=['statistics']).join(home_stats_df)
away_stats_df = pd.json_normalize(away_players['statistics'])
away_stats_df.index = away_players.index
away_players = away_players.drop(columns=['statistics']).join(away_stats_df)
home_players = home_players.reindex(columns=final_cols)
away_players = away_players.reindex(columns=final_cols)

In [351]:
home_player_flat = pd.json_normalize(home_players['player'])
home_players['Name'] = home_player_flat['name'].values
home_players['IdPlayer'] = home_player_flat['id'].values

home_team_ids = home_players[['IdPlayer', 'teamId']]
final_home_players = home_players.drop(columns=['player', 'teamId'])

# same for away
away_player_flat = pd.json_normalize(away_players['player'])
away_players['Name'] = away_player_flat['name'].values
away_players['IdPlayer'] = away_player_flat['id'].values

away_team_ids = away_players[['IdPlayer', 'teamId']]
final_away_players = away_players.drop(columns=['player', 'teamId'])
final_away_players.columns

Index(['jerseyNumber', 'position', 'substitute', 'captain', 'totalPass',
       'accuratePass', 'totalLongBalls', 'accurateLongBalls', 'goalAssist',
       'accurateOwnHalfPasses', 'totalOwnHalfPasses',
       'totalOppositionHalfPasses', 'totalClearance', 'ballRecovery',
       'goodHighClaim', 'totalKeeperSweeper', 'accurateKeeperSweeper',
       'minutesPlayed', 'touches', 'possessionLostCtrl', 'expectedAssists',
       'totalBallCarriesDistance', 'ballCarriesCount', 'totalProgression',
       'progressiveBallCarriesCount', 'totalShots', 'goalsPrevented',
       'passValueNormalized', 'dribbleValueNormalized',
       'defensiveValueNormalized', 'goalkeeperValueNormalized',
       'accurateOppositionHalfPasses', 'totalCross', 'aerialLost', 'aerialWon',
       'duelLost', 'duelWon', 'totalContest', 'wonContest', 'interceptionWon',
       'lastManTackle', 'totalTackle', 'wonTackle', 'unsuccessfulTouch',
       'fouls', 'topSpeed', 'kilometersCovered', 'numberOfSprints',
       'metersC

In [348]:
home_pf = pd.json_normalize(home_players['player']) if 'player' in home_players else home_player_flat
away_pf = pd.json_normalize(away_players['player']) if 'player' in away_players else away_player_flat

all_player_info = pd.concat([home_pf, away_pf], ignore_index=True)

players = pd.DataFrame({
    'IdPlayer': all_player_info['id'],
    'Name': all_player_info['name'],
    'Country': all_player_info['country.name'],
    'marketValue': all_player_info['proposedMarketValueRaw.value'],
    'dateOfBirth': pd.to_datetime(all_player_info['dateOfBirthTimestamp'], unit='s'),
    'height': all_player_info['height']
})

players = players.drop_duplicates(subset='IdPlayer').reset_index(drop=True)
all_team_ids = pd.concat([home_team_ids, away_team_ids], ignore_index=True)
players = players.merge(all_team_ids, on='IdPlayer', how='left')
players

,IdPlayer,Name,Country,marketValue,dateOfBirth,height,teamId
0,791092,Lawrence Ati Zigi,Ghana,1600000,1996-11-29,188,2442
1,1087805,Marvin Senaya,Ghana,3100000,2001-01-28,179,1646
2,1218056,Jonas Adjei Adjetey,Ghana,6400000,2003-12-13,188,2524
3,920514,Jerome Opoku,Ghana,5400000,1998-10-14,194,3086
4,844942,Gideon Mensah,Ghana,3700000,1998-07-18,177,1646
5,1986794,Caleb Yirenkyi,Ghana,10400000,2006-01-15,182,1292
6,904424,Elisha Owusu,Ghana,4300000,1997-11-07,179,1646
7,1019442,Kamaldeen Sulemana,Ghana,16500000,2002-02-15,174,2686
8,1202024,Ernest Nuamah,Ghana,9500000,2003-11-01,178,1649
9,103045,Jordan Ayew,Ghana,1100000,1991-09-11,182,31


In [312]:
highlights_str = row["highlights"].iloc[0]
highlights_json = json.loads(highlights_str)
highlights = pd.DataFrame(highlights_json)
key_subtitles = ['Goal', 'Goal (replay)', 'Chance', 'Chance (replay)', 'Big chance', 'Big chance (replay)', 'Cross', 'Goal Disallowed', 'Goal Disallowed (replay)'
                 'Penalty', 'Penalty (replay)', 'Penalty missed', 'VAR (Replay)', 'Penalty Disallowed (VAR decision)', 'Penalty Disallowed']
highlights_df = pd.DataFrame(highlights['highlights'].tolist()).sort_values('createdAtTimestamp')  # the actual content
highlights = highlights_df.loc[(highlights_df['subtitle'].isin(key_subtitles) | highlights_df['keyHighlight']), ['title', 'subtitle', 'url', 'createdAtTimestamp']]

highlights

,title,subtitle,url,createdAtTimestamp
1,Waterman C. (PAN) 2',Big chance,https://www.sofascore.com/video-player.html?ur...,1781737349
2,Waterman C. (PAN) 2',Big chance (replay),https://www.sofascore.com/video-player.html?ur...,1781737414
3,Blackman C. (PAN) 5',Chance,https://www.sofascore.com/video-player.html?ur...,1781737510
4,Martinez C. (PAN) 34',Chance,https://www.sofascore.com/video-player.html?ur...,1781739244
6,Martinez C. (PAN) 34',Chance (replay),https://www.sofascore.com/video-player.html?ur...,1781739317
7,Ramos J. (PAN) 38',Chance,https://www.sofascore.com/video-player.html?ur...,1781739465
8,Ramos J. (PAN) 38',Chance (replay),https://www.sofascore.com/video-player.html?ur...,1781739490
9,Murillo M. (PAN) 40',Chance,https://www.sofascore.com/video-player.html?ur...,1781739591
10,Murillo M. (PAN) 40',Chance (replay),https://www.sofascore.com/video-player.html?ur...,1781739633
11,Senaya M. (GHA) 45',Chance,https://www.sofascore.com/video-player.html?ur...,1781739913


In [ ]:
avg_positions = row["average_positions"].iloc[0]
avg_positions = json.loads(avg_positions)
avg_home_positions = pd.DataFrame(avg_positions['home'])
avg_away_positions = pd.DataFrame(avg_positions['away'])

,player,averageX,averageY,pointsCount
0,"{'name': 'Lawrence Ati Zigi', 'firstName': 'La...",8.008333,49.329167,24
1,"{'name': 'Kwasi Sibo', 'firstName': 'Kwasi', '...",42.913043,53.826087,23
2,"{'name': 'Benjamin Asare', 'firstName': 'Benja...",6.725000,47.937500,24
3,"{'name': 'Jordan Ayew', 'firstName': 'Jordan',...",63.220000,53.521818,55
4,"{'name': 'Prince Kwabena Adu', 'firstName': ''...",47.478571,82.614286,14
5,"{'name': 'Elisha Owusu', 'firstName': 'Elisha'...",48.626190,56.066667,42
6,"{'name': 'Ernest Nuamah', 'firstName': 'Ernest...",58.077143,17.251429,35
7,"{'name': 'Gideon Mensah', 'slug': 'gideon-mens...",39.773494,84.343373,83
8,"{'name': 'Kamaldeen Sulemana', 'firstName': ''...",58.906250,77.240625,32
9,"{'name': 'Abdul Fatawu Issahaku', 'firstName':...",62.851852,21.907407,27


In [288]:
def build_stats(groups, side):
    """side = 'home' or 'away'"""
    value_key = f'{side}Value'
    total_key = f'{side}Total'
    rows = []
    for group in groups:
        group_name = group['groupName']
        for item in group['statisticsItems']:
            name = item['name']
            if total_key in item:
                rows.append({'stat': f'Successful {group_name} {name}', 'value': item[value_key]})
                rows.append({'stat': f'Total {group_name} {name}', 'value': item[total_key]})
            else:
                rows.append({'stat': f'{group_name} {name}', 'value': item[value_key]})
    df = pd.DataFrame(rows).set_index('stat').T.reset_index(drop=True)
    return df



In [ ]:
statistics = row["statistics"].iloc[0]
statistics = json.loads(statistics)
statistics_df = pd.DataFrame(statistics['statistics'])
groups = statistics_df['groups'].iloc[0]
home_stats_df = build_stats(groups, 'home')
away_stats_df = build_stats(groups, 'away')

stat,Match overview Ball possession,Match overview Distance covered,Match overview Expected goals,Match overview Big chances,Match overview Total shots,Match overview Goalkeeper saves,Match overview Number of sprints,Match overview Corner kicks,Match overview Fouls,Match overview Passes,...,Defending Total tackles,Defending Interceptions,Defending Recoveries,Defending Clearances,Goalkeeping Total saves,Goalkeeping Goals prevented,Goalkeeping Big saves,Goalkeeping High claims,Goalkeeping Punches,Goalkeeping Goal kicks
0,38.0,87.21,1.25,1.0,7.0,4.0,85.0,2.0,9.0,352.0,...,16.0,8.0,50.0,28.0,4.0,1.3579,1.0,2.0,2.0,10.0


In [311]:
shotmaps = row["shotmap"].iloc[0]
shotmaps = json.loads(shotmaps)
shotmaps_df = pd.DataFrame(shotmaps['shotmap'])
shotmaps_df = shotmaps_df[['player', 'isHome', 'shotType', 'situation', 'playerCoordinates',
       'bodyPart', 'goalMouthLocation', 'goalMouthCoordinates',
       'blockCoordinates', 'xg', 'xgot', 'goalkeeper', 'time',
       'addedTime']]
